In [ ]:
# Cài pyspark
!pip install -q pyspark

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Khởi tạo Spark
spark = SparkSession.builder \
    .appName("RES_SYS") \
    .getOrCreate()

**Đọc dữ liệu User**

In [ ]:
# Đường dẫn file trong Google Drive
file = "/content/drive/MyDrive/BigData Amazon Data/Electronics.jsonl.gz"

In [ ]:
# Đọc file JSONL
df = spark.read.json(file)

In [ ]:
# Xem schema
df.printSchema()

In [ ]:
df.count(), df.distinct().count()

In [ ]:
df.select([
    count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
    if dict(df.dtypes)[c] in ['double', 'float']
    else count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

In [ ]:
df = df.dropDuplicates()

In [ ]:
df_train = spark.read.csv(r"/content/drive/MyDrive/BigData Amazon Data/Electronics.train.csv.gz",
                          header=True, inferSchema=True).select(
    "user_id",
    "parent_asin",
    "timestamp"
)

In [ ]:
df_val = spark.read.csv(r"/content/drive/MyDrive/BigData Amazon Data/Electronics.valid.csv.gz",
                          header=True, inferSchema=True).select(
    "user_id",
    "parent_asin",
    "timestamp"
)

In [ ]:
df_test = spark.read.csv(r"/content/drive/MyDrive/BigData Amazon Data/Electronics.test.csv.gz",
                          header=True, inferSchema=True).select(
    "user_id",
    "parent_asin",
    "timestamp"
)

In [ ]:
df_train_ok = broadcast(df_train).join(
    df,
    on=["user_id", "parent_asin", "timestamp"],
    how="left"
)

df_val_ok = broadcast(df_val).join(
    df,
    on=["user_id", "parent_asin", "timestamp"],
    how="left"
)

df_test_ok = broadcast(df_test).join(
    df,
    on=["user_id", "parent_asin", "timestamp"],
    how="left"
)

In [ ]:
df_train_ok.count(), df_val_ok.count(), df_test_ok.count()

In [ ]:
# =========================
# SAVE PARQUET
# =========================

df_train_ok.write \
    .mode("overwrite") \
    .parquet(
        "/content/drive/MyDrive/BigData Amazon Data/df_train_ok.parquet"
    )

df_val_ok.write \
    .mode("overwrite") \
    .parquet(
        "/content/drive/MyDrive/BigData Amazon Data/df_val_ok.parquet"
    )

df_test_ok.write \
    .mode("overwrite") \
    .parquet(
        "/content/drive/MyDrive/BigData Amazon Data/df_test_ok.parquet"
    )

**Đọc dữ liệu User Train - Val - Test**

In [ ]:
df_u_train = spark.read.parquet("/content/drive/MyDrive/BigData Amazon Data/Electronics_train.parquet")


In [ ]:
df_u_val = spark.read.parquet("/content/drive/MyDrive/BigData Amazon Data/Electronics_val.parquet")


In [ ]:
df_u_test = spark.read.parquet("/content/drive/MyDrive/BigData Amazon Data/Electronics_test.parquet")


**Đọc dữ liệu Item**

In [ ]:
# Đường dẫn file trong Google Drive
file = "/content/drive/MyDrive/BigData Amazon Data/meta_Electronics.jsonl.gz"

In [ ]:
schema = StructType([
    StructField("main_category", StringType()),
    StructField("title", StringType()),
    StructField("average_rating", DoubleType()),
    StructField("rating_number", IntegerType()),
    StructField("features", ArrayType(StringType())),
    StructField("description", ArrayType(StringType())),
    StructField("price", DoubleType()),
    StructField("images", ArrayType(StringType())),
    StructField("videos", ArrayType(StringType())),
    StructField("store", StringType()),
    StructField("categories", ArrayType(StringType())),

    # 👇 fix chỗ này
    StructField("details", MapType(StringType(), StringType())),

    StructField("parent_asin", StringType()),
    StructField("bought_together", ArrayType(StringType()))
])

df_i = spark.read.schema(schema).json(file)

In [ ]:
df_i.show(5)

In [ ]:
# Xem schema
df_i.printSchema()

In [ ]:
df_i.select([
    count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
    if dict(df_i.dtypes)[c] in ['double', 'float']
    else count(when(col(c).isNull(), c)).alias(c)
    for c in df_i.columns
]).show()

In [ ]:
from pyspark.sql import functions as F

# =========================================================
# DROP UNUSED COLUMNS
# =========================================================

df_i = df_i.drop("images", "videos", "bought_together")

# =========================================================
# ARRAY -> TEXT
# =========================================================

array_cols = [
    "features",
    "description",
    "categories"
]

for col_name in array_cols:
    df_i = df_i.withColumn(
        f"{col_name}_text",
        F.when(
            F.col(col_name).isNotNull(),
            F.concat_ws(" ", F.col(col_name))
        ).otherwise(F.lit(""))
    )

# =========================================================
# DETAILS MAP -> TEXT (NEW LINE FORMAT)
# =========================================================

df_i = df_i.withColumn(
    "details_text",
    F.when(
        F.col("details").isNotNull(),
        F.expr("""
            concat_ws(
                '\n',
                transform(
                    map_entries(details),
                    x -> concat(x.key, ': ', x.value)
                )
            )
        """)
    ).otherwise(F.lit(""))
)

# =========================================================
# TITLE NULL HANDLING
# =========================================================

df_i = df_i.withColumn(
    "title",
    F.when(
        F.col("title").isNull(),
        F.lit("")
    ).otherwise(F.col("title"))
)

# =========================================================
# BUILD SEMANTIC TEXT
# =========================================================

df_i = df_i.withColumn(
    "semantic_text",
    F.concat_ws(
        "\n\n",

        F.when(
            F.col("title") != "",
            F.concat(
                F.lit("TITLE\n"),
                F.col("title")
            )
        ),

        F.when(
            F.col("features_text") != "",
            F.concat(
                F.lit("FEATURES\n"),
                F.col("features_text")
            )
        ),

        F.when(
            F.col("description_text") != "",
            F.concat(
                F.lit("DESCRIPTION\n"),
                F.col("description_text")
            )
        ),

        F.when(
            F.col("categories_text") != "",
            F.concat(
                F.lit("CATEGORIES\n"),
                F.col("categories_text")
            )
        ),

        F.when(
            F.col("details_text") != "",
            F.concat(
                F.lit("DETAILS\n"),
                F.col("details_text")
            )
        )
    )
)

# =========================================================
# OPTIONAL NUMERICAL TRANSFORM
# =========================================================

df_i = df_i.withColumn(
    "price",
    F.when(
        F.col("price").isNotNull(),
        F.log1p(F.col("price"))
    )
)

df_i = df_i.withColumn(
    "rating_number",
    F.when(
        F.col("rating_number").isNotNull(),
        F.log1p(F.col("rating_number"))
    )
)

# =========================================================
# FINAL SELECT
# =========================================================

final_df_i = df_i.select(
    "parent_asin",
    "semantic_text",
    "main_category",
    "store",
    "price",
    "average_rating",
    "rating_number"
)

In [ ]:
final_df_i.show(20)

In [ ]:
rows = final_df_i.select("semantic_text").take(6)

print(rows[4]["semantic_text"])

In [ ]:
final_df_i.write \
    .mode("overwrite") \
    .parquet(
        "/content/drive/MyDrive/BigData Amazon Data/Electrics_item.parquet"
    )

In [ ]:
final_df_i.select([
    count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
    if dict(final_df_i.dtypes)[c] in ['double', 'float']
    else count(when(col(c).isNull(), c)).alias(c)
    for c in final_df_i.columns
]).show()

In [ ]:
df_u_train.show(5)

In [ ]:
df_u_train.select([
    count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
    if dict(df_u_train.dtypes)[c] in ['double', 'float']
    else count(when(col(c).isNull(), c)).alias(c)
    for c in df_u_train.columns
]).show()

In [ ]:
# Xem schema
df_u_train.printSchema()

In [ ]:
df_u_train.groupBy("rating").count().orderBy("rating").show()

In [ ]:
# Xem schema
final_df_i.printSchema()

**CODE**

In [ ]:
!pip install -q transformers sentence-transformers faiss-cpu pyarrow

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#user_df = pd.read_parquet("/content/drive/MyDrive/BigData Amazon Data/Electronics_train.parquet")

In [ ]:
!cp -r "/content/drive/MyDrive/BigData Amazon Data/Electronics_train.parquet" /content/

In [ ]:
import pandas as pd

user_df = pd.read_parquet(
    "/content/Electronics_train.parquet"
)

In [ ]:
item_df = pd.read_parquet("/content/drive/MyDrive/BigData Amazon Data/Electrics_item.parquet")

In [ ]:
print(item_df.head())
print(user_df.head())

In [ ]:
valid_items = set(item_df['parent_asin'])
user_df = user_df[user_df['parent_asin'].isin(valid_items)]